# AgriTech Cost Estimation — Phase 4

## Purpose

This notebook documents and validates the Cost Estimation methodology for the AgriTech project.

The production calculation engine is implemented separately in `ml/cost/calculator.py`. This notebook is for:

- documenting data requirements and sources
- validating units and transformations
- demonstrating how real source records are normalized
- combining valid yield data with mandi modal price
- calculating cultivation cost, revenue, net profit and ROI
- clearly distinguishing official data, user-provided inputs and calculated values

### Data integrity rule

This notebook does **not** create fake agricultural prices, yields, wages or input rates.

If an official source does not provide a required value, the value remains unavailable or is supplied explicitly by the user.

> The numerical records used in the small validation examples below are mathematical test values only; they are not presented as real agricultural observations.


## 1. Official data sources

### Mandi prices
Primary source: Government of India's Open Government Data platform / AGMARKNET.

The mandi dataset provides market-level daily commodity prices, including minimum, maximum and modal prices.

We use the **modal price** as the default market-price reference for revenue calculations.

### Crop recommendations and yield
Crop-specific agronomic recommendations and yield statistics should come from appropriate official agricultural sources such as ICAR and government agricultural statistics.

### Labour
Agricultural wage information should come from the Labour Bureau's rural wage statistics.

### Water requirement
Crop water/irrigation requirements should follow the FAO crop-water methodology. Irrigation cost itself is not assumed to be a universal government tariff.

### Important
Source availability varies by crop, region, season and year. Missing official values must not be replaced with invented values.


## 2. Imports

The notebook uses the Phase 3 loading/normalization layer and the Phase 1 deterministic calculator.

Expected project structure:

```text
AgriTech/
└── ml/
    └── cost/
        ├── __init__.py
        ├── calculator.py
        ├── constants.py
        ├── data_loader.py
        ├── normalizer.py
        └── validators.py
```


In [4]:
from pathlib import Path
import sys

# Find the AgriTech project root by looking for the ml/cost directory.
current_path = Path.cwd().resolve()

project_root = None

for path in [current_path, *current_path.parents]:
    if (path / "ml" / "cost").is_dir():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find the AgriTech project root. "
        "Make sure the notebook is inside the AgriTech project."
    )

ml_path = project_root / "ml"

if str(ml_path) not in sys.path:
    sys.path.insert(0, str(ml_path))

print("Project root:", project_root)
print("ML path:", ml_path)

from cost.calculator import calculate_cost
from cost.data_loader import load_csv, load_json, load_records
from cost.normalizer import (
    normalize_mandi_record,
    normalize_yield_record,
    normalize_labour_record,
    price_quintal_to_kg,
    hectare_to_acre,
)

print("Cost modules imported successfully.")

Project root: C:\Users\YUKTI\Desktop\projects\agri tech final\AgriTech
ML path: C:\Users\YUKTI\Desktop\projects\agri tech final\AgriTech\ml
Cost modules imported successfully.


## 3. Unit conventions

The production calculator expects:

- land area → acres
- seed rate → kg/acre
- seed price → INR/kg
- N/P/K rates → kg/acre
- fertilizer price → INR/kg
- water requirement → mm
- irrigation rate → INR/mm/acre when a paid rate is actually available
- labour requirement → days/acre
- wage → INR/day
- yield → kg/acre
- market price → INR/kg

Government datasets may use hectares, quintals or tonnes. Normalization converts those units before calculation.


In [5]:
# Mathematical unit checks — these are not agricultural observations.

assert price_quintal_to_kg(2500) == 25.0
assert round(hectare_to_acre(1), 10) == round(2.4710538147, 10)

print("Unit conversion checks passed.")


Unit conversion checks passed.


## 4. Loading source records

The Phase 3 loader supports CSV, JSON and already-loaded API-style records.

Example:

```python
records = load_csv("path/to/official_file.csv")
```

or:

```python
records = load_json("path/to/official_file.json")
```

When the official data API is available and integrated, its returned records can be passed to:

```python
records = load_records(api_records)
```

No fallback agricultural values are inserted by the loader.


In [6]:
# Loader interface smoke test using an empty record list.
# This does not represent agricultural data.

records = load_records([])
assert records == []

print("Data loader interface is ready for official records.")


Data loader interface is ready for official records.


## 5. Normalize mandi price records

AGMARKNET/data.gov.in-style records can contain prices in INR/quintal.

The normalizer preserves:

- crop/commodity
- state
- district
- market
- date
- minimum price
- maximum price
- modal price

and additionally converts available prices to INR/kg.

If a price is missing, it remains `None`.


In [7]:
# Mathematical validation record only — NOT real market data.

test_mandi = {
    "Commodity": "TEST_CROP",
    "State": "TEST_STATE",
    "District": "TEST_DISTRICT",
    "Market": "TEST_MARKET",
    "Arrival_Date": "2026-09-12",
    "Min_Price": "2000",
    "Max_Price": "3000",
    "Modal_Price": "2500",
}

mandi = normalize_mandi_record(test_mandi)

assert mandi["modal_price_inr_per_kg"] == 25.0
assert mandi["min_price_inr_per_kg"] == 20.0
assert mandi["max_price_inr_per_kg"] == 30.0

mandi


{'crop': 'TEST_CROP',
 'state': 'TEST_STATE',
 'district': 'TEST_DISTRICT',
 'market': 'TEST_MARKET',
 'date': '2026-09-12',
 'min_price_inr_per_quintal': 2000.0,
 'max_price_inr_per_quintal': 3000.0,
 'modal_price_inr_per_quintal': 2500.0,
 'min_price_inr_per_kg': 20.0,
 'max_price_inr_per_kg': 30.0,
 'modal_price_inr_per_kg': 25.0,
 'source': 'AGMARKNET/data.gov.in'}

In [8]:
# Missing-price integrity check.
missing_price_record = normalize_mandi_record({
    "Commodity": "TEST_CROP",
    "State": "TEST_STATE",
    "Market": "TEST_MARKET",
})

assert missing_price_record["modal_price_inr_per_kg"] is None

print("Missing mandi prices remain unavailable; no fallback price was invented.")


Missing mandi prices remain unavailable; no fallback price was invented.


## 6. Normalize official yield records

Yield is a separate agricultural variable from mandi price.

The relationship used for revenue is:

**Expected revenue = valid yield × valid mandi price**

Mandi data provides the price. It does not provide the farm's expected physical yield.

Yield records may be supplied in kg/hectare or tonnes/hectare and are normalized to kg/hectare and kg/acre.


In [9]:
# Mathematical validation record only — NOT real crop-yield data.

test_yield = normalize_yield_record({
    "Crop": "TEST_CROP",
    "State": "TEST_STATE",
    "Yield_kg_per_hectare": "2471.0538147",
})

assert round(test_yield["yield_kg_per_acre"], 6) == 1000.0

test_yield


{'crop': 'TEST_CROP',
 'state': 'TEST_STATE',
 'district': None,
 'season': None,
 'year': None,
 'yield_kg_per_hectare': 2471.0538147,
 'yield_kg_per_acre': 1000.0000000000001,
 'source': 'official'}

## 7. Normalize Labour Bureau records

Labour wages should not be reduced to one invented universal wage.

The normalizer keeps male and female wage observations separate because the correct labour-cost calculation depends on the labour composition used for the farm estimate.

The source record should retain state, year, month and agricultural activity/occupation so that the appropriate wage can be selected later.


In [10]:
# Mathematical validation record only — NOT real wage data.

test_labour = normalize_labour_record({
    "State": "TEST_STATE",
    "Year": "2026",
    "Month": "September",
    "Occupation": "Agricultural Labour",
    "Item": "Sowing",
    "Male": "400",
    "Female": "350",
})

assert test_labour["male_wage_inr_per_day"] == 400.0
assert test_labour["female_wage_inr_per_day"] == 350.0

test_labour


{'state': 'TEST_STATE',
 'year': '2026',
 'month': 'September',
 'occupation': 'Agricultural Labour',
 'activity': 'Sowing',
 'male_wage_inr_per_day': 400.0,
 'female_wage_inr_per_day': 350.0,
 'source': 'Labour Bureau'}

## 8. Water-cost policy

Water requirement and water cost are different concepts.

The project supports two situations:

### Free irrigation

If the farmer explicitly indicates that irrigation has no direct cost:

```text
water_cost = 0
```

This is a **user-provided condition**, not a government price.

### Paid irrigation

If irrigation has a direct cost, a verified or user-provided rate can be supplied.

The old prototype's hardcoded water rate is intentionally not used here because it was not an authoritative universal tariff.


## 9. Revenue methodology

For a valid yield and a valid mandi modal price:

```text
yield_kg_per_acre × modal_price_inr_per_kg
= expected revenue per acre
```

For a farm area:

```text
yield_kg_per_acre × land_size_acres
× modal_price_inr_per_kg
= expected revenue
```

If either yield or market price is unavailable, revenue should not be fabricated.


In [11]:
# End-to-end mathematical smoke test.
# ALL values below are synthetic unit-test values and must not be used as
# agricultural defaults or presented as official observations.

result = calculate_cost(
    crop="TEST_CROP",
    land_size_acres=1,
    seed_rate_kg_per_acre=10,
    seed_price_inr_per_kg=2,
    n_rate_kg_per_acre=1,
    p_rate_kg_per_acre=1,
    k_rate_kg_per_acre=1,
    fertilizer_price_inr_per_kg=3,
    water_requirement_mm=100,
    water_rate_inr_per_mm_per_acre=0,
    labor_days_per_acre=2,
    wage_inr_per_day=100,
    expected_yield_kg_per_acre=100,
    market_price_inr_per_kg=5,
)

expected = {
    "seed": 20.0,
    "fertilizer": 9.0,
    "water": 0.0,
    "labor": 200.0,
    "machinery": 0.0,
    "other_inputs": 0.0,
    "total_cost": 229.0,
    "estimated_revenue": 500.0,
    "net_profit": 271.0,
    "roi_percent": 118.34,
}

for key, value in expected.items():
    if key in result["breakdown"]:
        assert result["breakdown"][key] == value
    else:
        assert result[key] == value

result


{'crop': 'TEST_CROP',
 'land_size_acres': 1.0,
 'breakdown': {'seed': 20.0,
  'fertilizer': 9.0,
  'water': 0.0,
  'labor': 200.0,
  'machinery': 0.0,
  'other_inputs': 0.0},
 'total_cost': 229.0,
 'estimated_revenue': 500.0,
 'net_profit': 271.0,
 'roi_percent': 118.34}

## 10. Data provenance model

Every production value should eventually carry provenance.

Recommended categories:

| Value type | Example | Label |
|---|---|---|
| Government observation | AGMARKNET modal price | `official` |
| User input | Farmer's seed purchase price | `user_provided` |
| Derived value | INR/kg converted from INR/quintal | `calculated` |
| Historical observation | Previous mandi record | `historical` |
| Scenario assumption | Free irrigation selected by farmer | `user_assumption` |

This prevents the application from presenting assumptions as government facts.


## 11. Production data pipeline

The intended production flow is:

```text
Official source
      ↓
Raw record
      ↓
data_loader.py
      ↓
normalizer.py
      ↓
validators.py
      ↓
Validated normalized inputs
      ↓
ml/cost/calculator.py
      ↓
Cost + Revenue + Profit + ROI
```

The notebook is not the web-request execution layer. FastAPI will use the same calculation/data modules in later phases.


## 12. What is intentionally NOT in this notebook

- No IoT data
- No subsidy calculations
- No insurance calculations
- No fabricated historical mandi records
- No hardcoded mandi prices
- No universal hardcoded fertilizer price
- No universal hardcoded seed price
- No invented yield
- No fake confidence scores
- No meaningless `.pkl` model

The Cost Estimation module is a deterministic calculation engine, not an ML prediction model.


# Phase 4 completion checklist

- [x] Existing notebook replaced with a clean methodology notebook
- [x] Phase 1 calculator reused
- [x] Phase 3 loader reused
- [x] Unit conversions documented and tested
- [x] Mandi normalization documented and tested
- [x] Yield normalization documented and tested
- [x] Labour normalization documented and tested
- [x] Free-water policy documented
- [x] Yield × mandi-price revenue relationship documented
- [x] No fake agricultural dataset included
- [x] No production API key embedded
- [x] No `.pkl` created

**Next phase:** FastAPI integration after the official-data layer is connected appropriately.
